# Problem 3 — Catch the Eavesdropper (BB84)

Share a secret key over a simulated fiber link, then add an eavesdropper and catch them by the errors they cause.

**Your job:** fill in the encoding step and the eavesdropper, then report the error rate (QBER) with and without them.


In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()
rng = np.random.default_rng(42)

n_bits = 100          # number of qubits Alice sends


In [ ]:
# Alice's random bits and random bases (0 = Z/standard, 1 = X/Hadamard)
alice_bits  = rng.integers(0, 2, n_bits)
alice_bases = rng.integers(0, 2, n_bits)
bob_bases   = rng.integers(0, 2, n_bits)


In [ ]:
def encode(bit, basis):
    """Return a 1-qubit circuit preparing Alice's bit in the chosen basis."""
    qc = QuantumCircuit(1, 1)
    # TODO: if bit == 1, apply X. If basis == 1 (X basis), apply H.
    raise NotImplementedError("Prepare the qubit for this bit and basis")
    return qc


In [ ]:
def measure(qc, basis):
    """Measure a 1-qubit circuit in the given basis; return the classical bit."""
    if basis == 1:
        qc.h(0)                     # rotate X basis back to Z before measuring
    qc.measure(0, 0)
    counts = sim.run(transpile(qc, sim), shots=1).result().get_counts()
    return int(max(counts, key=counts.get))


In [ ]:
def run_bb84(eavesdrop=False):
    bob_results = []
    for i in range(n_bits):
        qc = encode(alice_bits[i], alice_bases[i])
        if eavesdrop:
            # TODO: Eve measures in a RANDOM basis, then re-prepares and resends.
            # Hint: pick eve_basis = rng.integers(0,2); eve_bit = measure(qc, eve_basis);
            #       then qc = encode(eve_bit, eve_basis) before Bob measures.
            raise NotImplementedError("Implement the intercept-resend eavesdropper")
        bob_results.append(measure(qc, bob_bases[i]))
    return np.array(bob_results)


In [ ]:
def qber(bob_results):
    """Sift (keep matching bases) and return the error rate on the shared key."""
    keep = alice_bases == bob_bases
    a = alice_bits[keep]
    b = bob_results[keep]
    if len(a) == 0:
        return 0.0
    return np.mean(a != b)

clean = run_bb84(eavesdrop=False)
print("QBER, no eavesdropper:", round(qber(clean), 3))

eve = run_bb84(eavesdrop=True)
print("QBER, with eavesdropper:", round(qber(eve), 3), "  (expect ~0.25)")


### Running on real quantum hardware (qBraid)

The cell below is a template. **Device IDs and the qBraid runtime API can change, so confirm both against your qBraid workspace** (the Devices tab, or `qbraid.runtime`) before you rely on it. Develop on the simulator first; use a real QPU only for the hardware deliverable, and remember the one-hardware-run-per-team budget.


In [ ]:
# --- Submit to a real QPU via qBraid ---
# Confirm the import and device ID against your workspace before running.
#
# from qbraid.runtime import QbraidProvider
# provider = QbraidProvider()
# for d in provider.get_devices():
#     print(d.id, d.num_qubits)          # find a device + its qubit count
# device = provider.get_device("<qpu_device_id>")   # e.g. an IonQ device
# job = device.run(qc, shots=200)
# counts = job.result().data.get_counts()
# print(counts)


## What to submit
- **Part 1:** a matching key with near-zero QBER (no eavesdropper).
- **Part 2:** the QBER with and without the eavesdropper (expect ~25%).
- **Part 3:** a hardware QBER, plus your threshold reasoning (in the response template).
- **Part 4:** the written physics explanation.
